# Agrupamiento de Canciones Spotify por Características de Audio
## Caso de Uso de Clustering — CRISP-DM
**Universidad Popular del Cesar**

**Docente:** Aimer Rivera Centeno  
**Metodología:** CRISP-DM  
**Dataset:** Spotify Tracks Dataset (114,000 canciones, 125 géneros)

In [ ]:
# ============================================================================
# CELDA 2 — IMPORTS
# ============================================================================
# Importamos todas las librerías necesarias para el análisis de clustering

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

# Preprocesamiento y clustering
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Persistencia de modelos
import joblib
import os

print('✓ Todas las librerías cargadas exitosamente')

In [ ]:
# ============================================================================
# CELDA 3 — CARGA Y ANÁLISIS EXPLORATORIO INICIAL (EDA)
# ============================================================================
# Cargamos el dataset de Spotify desde la carpeta data/

df = pd.read_csv('../data/dataset.csv')

# Vista general del dataset
print('Shape del dataset:', df.shape)
print('\nPrimeras 5 filas:')
display(df.head())

# Información de tipos de datos y nulos
print('\nInformación del dataset:')
df.info()

# Verificamos valores nulos por columna
print('\nValores nulos por columna:')
print(df.isnull().sum())

# Cantidad de géneros musicales únicos
print('\nGéneros únicos:', df['track_genre'].nunique())

In [ ]:
# ============================================================================
# CELDA 4 — DISTRIBUCIONES DE FEATURES DE AUDIO
# ============================================================================
# Visualizamos la distribución de cada característica de audio para entender
# su comportamiento y detectar posibles valores atípicos

features_audio = ['danceability', 'energy', 'loudness', 'speechiness',
                  'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()
for i, feat in enumerate(features_audio):
    axes[i].hist(df[feat].dropna(), bins=50, color='#1DB954', edgecolor='white', alpha=0.8)
    axes[i].set_title(f'Distribución: {feat}', fontsize=12)
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('Frecuencia')

plt.suptitle('Distribuciones de Características de Audio — Spotify', fontsize=16)
plt.tight_layout()
plt.show()

print('✓ Histogramas generados correctamente')

In [ ]:
# ============================================================================
# CELDA 5 — MAPA DE CORRELACIÓN INTERACTIVO
# ============================================================================
# Calculamos la matriz de correlación entre las features de audio
# para identificar relaciones lineales entre variables

corr_features = df[features_audio + ['popularity']].corr()

fig = px.imshow(corr_features,
                title='Matriz de Correlación — Features de Audio',
                color_continuous_scale='RdBu_r',
                text_auto='.2f',
                width=800, height=700)
fig.show()

print('\nObservaciones:')
print('- Energy y loudness tienen correlación positiva fuerte (r > 0.7)')
print('- Energy y acousticness tienen correlación negativa fuerte (r < -0.7)')
print('- Estos patrones son consistentes con la naturaleza de la música')

In [ ]:
# ============================================================================
# CELDA 6 — SCATTER DANCEABILITY vs ENERGY POR GÉNERO
# ============================================================================
# Tomamos una muestra de 3000 canciones para visualizar la relación entre
# danceability y energy, coloreada por género musical

muestra = df.sample(3000, random_state=42)

fig = px.scatter(muestra, x='danceability', y='energy',
                 color='track_genre',
                 hover_data=['track_name', 'artists'],
                 title='Danceability vs Energy por Género (muestra 3,000 canciones)',
                 opacity=0.6,
                 width=900, height=600)
fig.show()

print('✓ Scatter plot generado con 3,000 muestras aleatorias')

In [ ]:
# ============================================================================
# CELDA 7 — PREPARACIÓN DE DATOS PARA CLUSTERING
# ============================================================================
# Seleccionamos las 10 features numéricas y aplicamos normalización
# con StandardScaler para que todas las variables tengan el mismo peso

features_cluster = ['danceability', 'energy', 'loudness', 'speechiness',
                    'acousticness', 'instrumentalness', 'liveness',
                    'valence', 'tempo', 'popularity']

# Eliminamos filas con valores nulos
df_cluster = df[features_cluster].dropna()
print(f'Registros para clustering: {df_cluster.shape[0]:,}')
print(f'Features seleccionadas: {len(features_cluster)}')

# Normalizamos los datos (media=0, desviación=1)
# Este paso es CRÍTICO porque K-Means usa distancias euclidianas y
# variables con rangos grandes (tempo ~200, loudness ~-60) dominarían
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_cluster)
print(f'Datos normalizados. Shape: {X_scaled.shape}')
print('Media después de escalar:', X_scaled.mean(axis=0).round(2))
print('Desviación después de escalar:', X_scaled.std(axis=0).round(2))

In [ ]:
# ============================================================================
# CELDA 8 — MÉTODO DEL CODO Y SILHOUETTE PARA K ÓPTIMO
# ============================================================================
# Evaluamos K desde 2 hasta 10 usando dos métricas:
# 1. Inercia (método del codo) — busca el punto de inflexión
# 2. Silhouette Score — mide cohesión y separación de los clusters

inertias = []
silhouettes = []
K_range = range(2, 11)

print('Evaluando K de 2 a 10...\n')
for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)
    
    # Silhouette score con muestra para eficiencia (114k registros)
    sil = silhouette_score(X_scaled, kmeans.labels_, sample_size=5000, random_state=42)
    silhouettes.append(sil)
    print(f'K={k} | Inertia: {kmeans.inertia_:,.0f} | Silhouette: {sil:.4f}')

# Gráfica 1: Método del Codo
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(K_range, inertias, 'bo-')
axes[0].set_title('Método del Codo', fontsize=14)
axes[0].set_xlabel('Número de Clusters (K)')
axes[0].set_ylabel('Inercia')
axes[0].grid(True, alpha=0.3)

# Gráfica 2: Silhouette Score
axes[1].plot(K_range, silhouettes, 'rs-')
axes[1].set_title('Silhouette Score por K', fontsize=14)
axes[1].set_xlabel('Número de Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Identificamos el K óptimo según silhouette
k_optimo_sil = K_range[np.argmax(silhouettes)]
print(f'\n→ K con mayor Silhouette Score: K={k_optimo_sil} ({silhouettes[k_optimo_sil-2]:.4f})')
print('→ Revisar el método del codo para confirmar el punto de inflexión')

In [ ]:
# ============================================================================
# CELDA 9 — K-MEANS CON K ÓPTIMO
# ============================================================================
# Entrenamos K-Means con el número de clusters óptimo.
# K=6 es el valor por defecto (ajustar según resultados del codo/silhouette)

K_OPTIMO = 6  # <--- AJUSTAR SEGÚN RESULTADOS DE CELDA 8

kmeans = KMeans(n_clusters=K_OPTIMO, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

# Agregamos las etiquetas al DataFrame
df_cluster = df_cluster.copy()
df_cluster['cluster'] = clusters
df.loc[df_cluster.index, 'cluster'] = clusters

# Evaluamos la calidad del clustering
sil_final = silhouette_score(X_scaled, clusters, sample_size=5000, random_state=42)
db_final = davies_bouldin_score(X_scaled, clusters)

print(f'K-Means con K={K_OPTIMO}')
print('=' * 40)
print(f'Silhouette Score: {sil_final:.4f}')
print(f'Davies-Bouldin Index: {db_final:.4f}')
print(f'\nDistribución de canciones por cluster:')
print(pd.Series(clusters).value_counts().sort_index())

In [ ]:
# ============================================================================
# CELDA 10 — VISUALIZACIÓN PCA 2D DE LOS CLUSTERS
# ============================================================================
# Reducimos dimensionalidad a 2 componentes principales para visualizar
# cómo se separan los clusters en el espacio reducido

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

varianza_explicada = pca.explained_variance_ratio_.sum() * 100
print(f'Varianza explicada por 2 componentes PCA: {varianza_explicada:.1f}%')
print(f'Componente 1: {pca.explained_variance_ratio_[0]*100:.1f}%')
print(f'Componente 2: {pca.explained_variance_ratio_[1]*100:.1f}%')

# DataFrame para graficar
df_plot = pd.DataFrame({
    'PC1': X_pca[:, 0],
    'PC2': X_pca[:, 1],
    'Cluster': clusters.astype(str)
})

fig = px.scatter(df_plot, x='PC1', y='PC2', color='Cluster',
                 title=f'Clusters K-Means (K={K_OPTIMO}) — Proyección PCA 2D',
                 opacity=0.5,
                 color_discrete_sequence=px.colors.qualitative.Set1,
                 width=900, height=600)
fig.show()

In [ ]:
# ============================================================================
# CELDA 11 — PERFIL DE CADA CLUSTER
# ============================================================================
# Calculamos el promedio de cada feature por cluster para entender
# qué tipo de música representa cada grupo

perfil = df_cluster.groupby('cluster')[features_cluster].mean().round(3)
print('Perfil promedio por cluster:')
print('=' * 80)
display(perfil)

# --- RADAR CHART ---
# Gráfico radial que muestra el perfil de audio de cada cluster
fig = go.Figure()

# Seleccionamos features para el radar (las que están en escala 0-1)
features_radar = ['danceability', 'energy', 'speechiness',
                  'acousticness', 'liveness', 'valence']

for cluster_id in range(K_OPTIMO):
    valores = perfil.loc[cluster_id, features_radar].tolist()
    valores += [valores[0]]  # Cerramos el polígono
    fig.add_trace(go.Scatterpolar(
        r=valores,
        theta=features_radar + [features_radar[0]],
        fill='toself',
        name=f'Cluster {cluster_id}'
    ))

fig.update_layout(
    title='Perfil de Audio por Cluster (Radar Chart)',
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    width=800, height=600
)
fig.show()

# --- NOMBRES SUGERIDOS PARA CLUSTERS ---
# Basados en el perfil promedio, asignamos nombres descriptivos
# INSTRUCCIÓN: Ajustar estos nombres después de revisar el perfil real
nombres_clusters = {
    0: '🎵 Energía y Baile',
    1: '🎵 Acústico y Relajado',
    2: '🎵 Instrumental y Experimental',
    3: '🎵 Hablado y Podcast',
    4: '🎵 En Vivo y Energético',
    5: '🎵 Melódico y Positivo'
}

print('\n' + '=' * 50)
print('NOMBRES SUGERIDOS PARA CADA CLUSTER')
print('=' * 50)
print('(Ajustar según el perfil real observado en la tabla superior)')
for k, v in nombres_clusters.items():
    print(f'  Cluster {k}: {v}')

In [ ]:
# ============================================================================
# CELDA 12 — GUARDAR MODELOS PARA LA APP FLASK
# ============================================================================
# Guardamos los 3 artefactos que necesita la aplicación web:
# 1. kmeans_spotify.pkl  → modelo K-Means entrenado
# 2. scaler_spotify.pkl   → scaler para normalizar inputs
# 3. pca_spotify.pkl      → PCA para visualización 2D

# Ruta de salida: app/agrupamiento_app/model/
model_dir = '../../../app/agrupamiento_app/model'
os.makedirs(model_dir, exist_ok=True)

joblib.dump(kmeans, f'{model_dir}/kmeans_spotify.pkl')
joblib.dump(scaler, f'{model_dir}/scaler_spotify.pkl')
joblib.dump(pca, f'{model_dir}/pca_spotify.pkl')

print('✓ Modelos guardados exitosamente en:')
print(f'  {model_dir}/')
print(f'    - kmeans_spotify.pkl  ({os.path.getsize(f"{model_dir}/kmeans_spotify.pkl")/1024:.1f} KB)')
print(f'    - scaler_spotify.pkl  ({os.path.getsize(f"{model_dir}/scaler_spotify.pkl")/1024:.1f} KB)')
print(f'    - pca_spotify.pkl     ({os.path.getsize(f"{model_dir}/pca_spotify.pkl")/1024:.1f} KB)')

## Conclusiones

- Se identificaron **K clusters** con características de audio diferenciadas, cada uno representando un perfil musical distinto.
- El **Silhouette Score** de **[VALOR]** indica que los clusters están **[bien/moderadamente]** definidos, con buena separación entre grupos.
- Los clusters se caracterizan por:
  - **Cluster 0 (Energía y Baile):** alta danceability y energy, ideal para música electrónica y pop.
  - **Cluster 1 (Acústico y Relajado):** alta acousticness, baja energy, canciones acústicas.
  - **Cluster 2 (Instrumental):** alta instrumentalness, poca voz, bandas sonoras.
  - **Cluster 3 (Hablado):** alta speechiness, podcasts o rap.
  - **Cluster 4 (En Vivo):** alta liveness, grabaciones en concierto.
  - **Cluster 5 (Melódico):** alta valence y danceability, música positiva.
- **Aplicación práctica:** Este modelo puede integrarse en un sistema de recomendación que sugiera canciones del mismo cluster al usuario.
- **Trabajo futuro:** Explorar clustering jerárquico para subgéneros y probar DBSCAN para detectar outliers musicales.